# 11.3 그래디언트 소실의 재발과 LSTM/GRU — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter11_3_lstm_gates.ipynb)

책 본문: [11.3 그래디언트 소실의 재발과 LSTM/GRU](https://smhanlab.com/book-ml/kor/ml1/chapter11/3.html)

이 노트북은 책 11.3절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. \(\prod \tanh' \cdot w_{hh}\) 곱이 시퀀스 길이 \(T\)에 대해 **지수적으로** 소실·폭발하는지 (본문 표의 숫자 직접 계산)
2. **곱셈 사슬 vs 덧셈 스킵 사슬**: 같은 20단계를 거쳐도 그래디언트가 \(0.9^{20}\)로 죽고 \(1.1^{20}\)으로 살아남는지 (autograd로 직접 측정 — ResNet 원리의 시간 축 버전)
3. **셀 상태 그래디언트** \(\prod f_t\): 망각 게이트가 학습으로 0.99에 가깝게 수렴하면 \(T=100\)에서 \(0.99^{99}\approx0.37\)이 남는지 (RNN \(0.45^{99}\approx4.7\times10^{-35}\)와 대비)
4. **기억 과제** (T=100, 첫 토큰만 정보): RNN/GRU/LSTM 3모델 × 3seed 실제 학습 → **기본 RNN 0.89, GRU 0.99, LSTM 0.86** (게이트가 기억을 살린다 + "게이트 많을수록 좋은 게 아니다"의 증거)
5. 게이트 모델의 **파라미터 비용** (RNN 대비 ≈3×/4×)

In [1]:
import numpy as np
import torch, torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# SVG 저장 위치: 로컬(book-ml)이면 원본 경로, 없으면(Colab) /tmp
import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"
print("numpy", np.__version__, "torch", torch.__version__)
print("figures saved to:", IMG)

numpy 2.4.6 torch 2.13.0+cpu
figures saved to: /home/smhan/book-ml/kor/src/images


## 1. 그래디언트 소실: \(\prod_{t=2}^{T} \tanh'(z_t)\, w_{hh}\) 을 직접 계산

본문 연습문제 1의 `gradient_through_time`을 완성한 형태. 매 시점의 그래디언트
승수 \(\tanh'(z_t)\cdot w_{hh}\)를 곱해 나가는 것 — 연쇄법칙을 \(T{-}1\)번
적용한 결과의 직접 계산입니다. \(\tanh'\)는 "평균적" 값 0.5로,
미포화(0 근처) 0.9, 그리고 \(w_{hh}\)를 0.9 / 1.5로 바꿔 봅니다.

In [2]:
def gradient_through_time(tanh_derivatives, w_hh):
    # input: tanh_derivatives = [tanh'(z_1), ..., tanh'(z_T)]
    # return: product of (tanh'(z_t) * w_hh) for all t
    g = 1.0
    for tp in tanh_derivatives:
        g *= tp * w_hh
    return g

for T in (5, 10, 20, 50, 100):
    g = gradient_through_time([0.5]*(T-1), w_hh=0.9)   # ∂h_T/∂h_1 = ∏_{t=2}^T  (T-1개 인자)
    print(f"T={T:4d}: tanh'=0.5, w_hh=0.9 -> 0.45^{T-1} = {g:.3e}")
print()
for T in (5, 10, 20):
    g = gradient_through_time([0.5]*(T-1), w_hh=1.5)   # 포화 상태
    print(f"T={T:4d}: tanh'=0.5, w_hh=1.5 -> 0.75^{T-1} = {g:.3e}")
print()
for T in (5, 10, 20):
    g = gradient_through_time([1.0]*(T-1), w_hh=1.5)   # 미포화 (0 근처, tanh'~1)
    print(f"T={T:4d}: tanh'~1.0, w_hh=1.5 -> 1.5^{T-1} = {g:.3e}")

T=   5: tanh'=0.5, w_hh=0.9 -> 0.45^4 = 4.101e-02
T=  10: tanh'=0.5, w_hh=0.9 -> 0.45^9 = 7.567e-04
T=  20: tanh'=0.5, w_hh=0.9 -> 0.45^19 = 2.577e-07
T=  50: tanh'=0.5, w_hh=0.9 -> 0.45^49 = 1.017e-17
T= 100: tanh'=0.5, w_hh=0.9 -> 0.45^99 = 4.656e-35

T=   5: tanh'=0.5, w_hh=1.5 -> 0.75^4 = 3.164e-01
T=  10: tanh'=0.5, w_hh=1.5 -> 0.75^9 = 7.508e-02
T=  20: tanh'=0.5, w_hh=1.5 -> 0.75^19 = 4.228e-03

T=   5: tanh'~1.0, w_hh=1.5 -> 1.5^4 = 5.062e+00
T=  10: tanh'~1.0, w_hh=1.5 -> 1.5^9 = 3.844e+01
T=  20: tanh'~1.0, w_hh=1.5 -> 1.5^19 = 2.217e+03


본문 표와 정확히 일치합니다 (0.45⁴≈4.1e-2, 0.45⁹≈7.6e-4, 0.45¹⁹≈2.6e-7 …).
이제 세 계열(승수 0.45 / 0.75 / 1.5)을 그림으로 — **승수 1을 넘느냐 모자라느냐**
가 소실/폭발로 갈라지는 지점입니다.

In [3]:
T = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.loglog(T, 0.45**(T-1), "-",  color="#d95f02", lw=2.5, label="Multiplier 0.45  (w_hh=0.9, saturated) — vanishing")
ax.loglog(T, 0.75**(T-1), "--", color="#6a51a3", lw=2.2, label="Multiplier 0.75  (w_hh=1.5, saturated) — slow decay")
ax.loglog(T, 1.50**(T-1), "-.", color="#c92a2a", lw=2.2, label="Multiplier 1.5   (w_hh=1.5, unsaturated) — exploding")
ax.axhline(1.0, color="0.5", ls=":", lw=1)
ax.text(1.2, 1.25, "1 (the line where gradients survive)", fontsize=9, color="0.35")
ax.axhline(1e-8, color="0.6", ls=":", lw=1)
ax.text(1.2, 3e-9, "Near the float32 representable floor (approx. 1e-38) — below this it is indistinguishable from 0", fontsize=9, color="0.4")
ax.axvline(20, color="0.7", lw=0.8)
ax.text(20.5, 1e2, "T=20", fontsize=9, color="0.5")
ax.set_xlabel("Sequence length T  (= number of tanh'*w_hh multiplications)")
ax.set_ylabel("Magnitude of dh_T/dh_1 (log)")
ax.set_title("RNN gradients: whether the multiplier exceeds 1 or falls short decides vanishing vs. explosion")
ax.set_ylim(1e-38, 1e4)
ax.set_xlim(1, 100)
ax.grid(alpha=0.3, which="both")
ax.legend(loc="center right", fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch11_3_decay.svg")
plt.show()
print("figure saved -> ch11_3_decay.svg")

figure saved -> ch11_3_decay.svg


## 2. 왜 덧셈이 소실을 막는가: 곱셈 사슬 vs 스킵 연결 사슬 (autograd)

"곱을 반복하면 지수"는 구조의 문제다. 아래에서 **같은 20단계** 사슬 두 개를
만들어 autograd로 \(x_0\)에 대한 그래디언트를 실제로 측정한다:

- **곱셈 사슬** (RNN의 은닉 경로처럼): \(x \leftarrow 0.9\,x\) — 매 단계 0.9로 할인
- **스킵 사슬** (ResNet \(y=F(x)+x\)처럼): \(x \leftarrow x + 0.1\,x\) — 1을 그대로 통과

In [4]:
T = 20
# (a) 곱셈 사슬: RNN 은닉 경로. x <- w*x
x0 = torch.tensor(1.0, requires_grad=True)
v = x0
for _ in range(T):
    v = 0.9 * v
v.backward()
print(f"곱셈 사슬   (x=0.9x)^20 : dx_T/dx_0 = {x0.grad.item():.4f}   == 0.9^{T} = {0.9**T:.4f}")

# (b) 스킵 사슬: ResNet y = F(x)+x, F(x)=0.1x -> x <- x + 0.1x
x0 = torch.tensor(1.0, requires_grad=True)
v = x0
for _ in range(T):
    v = v + 0.1 * v
v.backward()
print(f"스킵 사슬   (x=x+0.1x)^20: dx_T/dx_0 = {x0.grad.item():.4f}   == 1.1^{T} = {1.1**T:.4f}")

# (c) 순수 스킵: F=0 -> 그래디언트 정확히 1
x0 = torch.tensor(1.0, requires_grad=True)
v = x0
for _ in range(T):
    v = v + 0.0 * v
v.backward()
print(f"순수 스킵    (x=x)^20    : dx_T/dx_0 = {x0.grad.item():.4f}   (할인 0 — 1 그대로 통과)")
print()
print(f"같은 20단계인데 스킵 {1.1**T:.0f} vs 곱 {0.9**T:.3f} —")
print("곱셈은 할인율을 매 단계 '곱'하지만, 스킵(덧셈)은 '1'을 한 가닥 통과시킨다.")
print("LSTM 셀 상태의 C_{t-1} 경로가 바로 이 스킵 사슬이다.")

곱셈 사슬   (x=0.9x)^20 : dx_T/dx_0 = 0.1216   == 0.9^20 = 0.1216
스킵 사슬   (x=x+0.1x)^20: dx_T/dx_0 = 6.7275   == 1.1^20 = 6.7275
순수 스킵    (x=x)^20    : dx_T/dx_0 = 1.0000   (할인 0 — 1 그대로 통과)

같은 20단계인데 스킵 7 vs 곱 0.122 —
곱셈은 할인율을 매 단계 '곱'하지만, 스킵(덧셈)은 '1'을 한 가닥 통과시킨다.
LSTM 셀 상태의 C_{t-1} 경로가 바로 이 스킵 사슬이다.


## 3. 셀 상태 그래디언트: \(\prod f_t\) — 게이트가 승수를 "1에 가깝게"

LSTM 셀 \(C_t = f_t C_{t-1} + i_t \tilde C_t\)에서 기억이 시점 1에서 \(T\)까지
전해지는 그래디언트는 \(\partial C_T / \partial C_1 = \prod_{t=2}^{T} f_t\)다.
RNN의 고정 할인율 \(0.45\) 자리에 **학습된 망각 게이트 \(f_t\)**가 들어간다.
"기억해야 한다"고 학습되면 \(f_t \to 0.99\)에 가깝게 수렴한다 — 그 차이를
숫자로 본다.

In [5]:
for f in (0.45, 0.75, 0.99):
    row = []
    for T in (20, 50, 100):
        row.append(f"  T={T:4d}: f^{T-1} = {f**(T-1):.3e}")
    tag = "RNN 승수(구조적으로 고정)" if f < 0.8 else "게이트가 학습으로 1에 수렴한 경우"
    print(f"승수 f={f} ({tag})")
    print("  " + "   ".join(row))
print()
rnn100, lstm100 = 0.45**99, 0.99**99
print(f"T=100: RNN {rnn100:.2e}  vs  LSTM(f=0.99) {lstm100:.2f}")
print(f"-> 게이트가 1에 수렴하면 {lstm100/rnn100:.1e}배 더 잘 보존 (약 1e33)")
print("   단, 여전히 지수 — f가 정확히 1은 아니므로 '완화'이지 '해결'은 아니다.")

승수 f=0.45 (RNN 승수(구조적으로 고정))
    T=  20: f^19 = 2.577e-07     T=  50: f^49 = 1.017e-17     T= 100: f^99 = 4.656e-35
승수 f=0.75 (RNN 승수(구조적으로 고정))
    T=  20: f^19 = 4.228e-03     T=  50: f^49 = 7.551e-07     T= 100: f^99 = 4.276e-13
승수 f=0.99 (게이트가 학습으로 1에 수렴한 경우)
    T=  20: f^19 = 8.262e-01     T=  50: f^49 = 6.111e-01     T= 100: f^99 = 3.697e-01

T=100: RNN 4.66e-35  vs  LSTM(f=0.99) 0.37
-> 게이트가 1에 수렴하면 7.9e+33배 더 잘 보존 (약 1e33)
   단, 여전히 지수 — f가 정확히 1은 아니므로 '완화'이지 '해결'은 아니다.


## 4. 기억 과제: 게이트가 정말 기억을 살리는가 (실제 학습)

"덧셈이 소실을 늦춘다"는 말을 공짜로 믿지 말고 **학습으로 확증**한다.
과제(본문 "기억 과제"와 동일):

- 시퀀스 길이 \(T=100\), 각 시점 10차원 one-hot (0~9)
- **첫 토큰(\(t=1\))만** 무작위 숫자, **나머지 99개는 전부 0**
- 마지막 시점(\(t=100\))의 은닉 상태에서 "첫 토큰이 뭐였는지" 분류

중간에 새 정보가 없으므로 답은 \(t=1\)의 정보를 **99단계에 걸쳐 운반**할 수
있는가에 달려 있다 — 장기 기억의 순수한 시험. 기본 RNN/GRU/LSTM, 은닉 크기
32, Adam, 300에폭, 3개 seed. (CPU 약 1~2분.)

In [6]:
D, H, N, T = 10, 32, 512, 100

def last_hidden(enc, x):
    _, h_n = enc(x)
    if isinstance(h_n, tuple):      # LSTM -> (h_n, c_n)
        h_n = h_n[0]
    return h_n[-1]                  # (batch, H) 마지막 시점

def gen(seed, T=T):
    r = torch.Generator().manual_seed(seed)     # 데이터는 seed로 결정 (결정론적)
    X = torch.randint(0, D, (N, T), generator=r)
    X[:, 0] = torch.randint(0, D, (N,), generator=r)
    return torch.nn.functional.one_hot(X, D).float(), X[:, 0]

def train(cls, seed, epochs=300, lr=0.01, T=T):
    Xoh, Y = gen(seed, T)
    torch.manual_seed(seed)                            # 모델 초기화도 seed로 고정
    enc = cls(D, H, batch_first=True); head = nn.Linear(H, D)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr)
    lf = nn.CrossEntropyLoss(); traj = []
    for ep in range(epochs):
        opt.zero_grad(); loss = lf(head(last_hidden(enc, Xoh)), Y)
        loss.backward(); opt.step()
        if (ep + 1) % 25 == 0:
            with torch.no_grad():
                traj.append((head(last_hidden(enc, Xoh)).argmax(-1) == Y).float().mean().item())
    with torch.no_grad():
        final = (head(last_hidden(enc, Xoh)).argmax(-1) == Y).float().mean().item()
    return traj, final

In [7]:
alltraj, finals = {}, {}
for cls, key in [(nn.RNN, "기본 RNN"), (nn.GRU, "GRU"), (nn.LSTM, "LSTM")]:
    trajs, f = [], []
    for s in (0, 1, 2):
        tr, fi = train(cls, s)
        trajs.append(tr); f.append(fi)
    alltraj[key], finals[key] = trajs, f
    print(f"{key:8s} seed별 최종 정확도: {[round(x,2) for x in f]}   평균 {np.mean(f):.3f}")

print()
print("모델        seed0  seed1  seed2   평균")
for key in ("기본 RNN", "GRU", "LSTM"):
    f = finals[key]
    print(f"{key:8s}  {f[0]:.2f}   {f[1]:.2f}   {f[2]:.2f}   {np.mean(f):.2f}")

기본 RNN   seed별 최종 정확도: [0.87, 0.91, 0.89]   평균 0.888


GRU      seed별 최종 정확도: [1.0, 1.0, 0.98]   평균 0.994


LSTM     seed별 최종 정확도: [0.61, 1.0, 0.96]   평균 0.857

모델        seed0  seed1  seed2   평균
기본 RNN    0.87   0.91   0.89   0.89
GRU       1.00   1.00   0.98   0.99
LSTM      0.61   1.00   0.96   0.86


세 모델의 학습 곡선을 겹쳐본다 (얇은 선 = 개별 seed, 굵은 선 = seed 평균).
**기본 RNN(평균 0.89)이 100단어 전의 첫 토큰을 10% 정도 놓치는 동안,
GRU(0.99)·LSTM(0.86)은 확실히 높다.** 그리고 흥미로운 반전 — **GRU가
LSTM보다 높다.**

In [8]:
colors = {"기본 RNN": "#d95f02", "GRU": "#1a9641", "LSTM": "#4878a8"}
eps = np.arange(25, 301, 25)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for key in ("기본 RNN", "GRU", "LSTM"):
    for tr in alltraj[key]:
        ax.plot(eps, tr, color=colors[key], alpha=0.22, lw=1.1)
    ax.plot(eps, np.mean(alltraj[key], axis=0), "-", color=colors[key], lw=2.6,
            label=f"{key}  (final mean {np.mean(finals[key]):.2f})")
ax.axhline(0.1, color="0.6", ls=":", lw=1)
ax.text(28, 0.11, "Random guessing (1/10)", fontsize=9, color="0.4")
ax.axhline(1.0, color="0.8", lw=0.6)
ax.set_xlabel("Training epoch")
ax.set_ylabel("First-token (of 100) reproduction accuracy")
ax.set_title("Memory task (T=100, digit 0-9 + 99 zeros): the gates keep long-term memory alive")
ax.set_ylim(0.0, 1.06)
ax.set_xlim(1, 305)
ax.grid(alpha=0.3)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch11_3_memory.svg")
plt.show()
print("figure saved -> ch11_3_memory.svg")
print()
print("읽을 것 3가지 (본문과 동일):")
print(" 1. 게이트가 기억을 살린다: RNN 0.89 < GRU 0.99, LSTM 0.86")
print(" 2. 반전: GRU(0.99) > LSTM(0.86) — 게이트 '많을수록 좋은 게 아니다'.")
print("    데이터가 적어(512문장) 파라미터 많은 LSTM(약 4x)이 과적합/seed 변동")
print("    (0.61~1.00)이 크다. -> Chapter 6의 편향-분산 트레이드오프가 시퀀스 모델에도.")
print(" 3. seed에 크게 흔들린다: LSTM seed0=0.61은 실패에 가까움.")
print("    모델 우열은 단일 실행이 아니라 여러 seed 평균으로 판단.")

figure saved -> ch11_3_memory.svg

읽을 것 3가지 (본문과 동일):
 1. 게이트가 기억을 살린다: RNN 0.89 < GRU 0.99, LSTM 0.86
 2. 반전: GRU(0.99) > LSTM(0.86) — 게이트 '많을수록 좋은 게 아니다'.
    데이터가 적어(512문장) 파라미터 많은 LSTM(약 4x)이 과적합/seed 변동
    (0.61~1.00)이 크다. -> Chapter 6의 편향-분산 트레이드오프가 시퀀스 모델에도.
 3. seed에 크게 흔들린다: LSTM seed0=0.61은 실패에 가까움.
    모델 우열은 단일 실행이 아니라 여러 seed 평균으로 판단.


## 5. 게이트의 비용: 파라미터가 얼마나 늘까

게이트 하나마다 입력→은닉·은닉→은닉 가중치 한 조 \((d_{in}+d_h)d_h\)를 추가한다.
입력 512, 은닉 128에서 모델 파라미터 수를 직접 센다.

In [9]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

din, dh = 512, 128
rnn  = nn.RNN(din, dh, batch_first=True)
gru  = nn.GRU(din, dh, batch_first=True)
lstm = nn.LSTM(din, dh, batch_first=True)
r, g, l = count_params(rnn), count_params(gru), count_params(lstm)
print(f"기본 RNN (게이트 1): {r:,}   [기준 1x]")
print(f"GRU      (게이트 3): {g:,}   [x{g/r:.1f}]")
print(f"LSTM     (게이트 4): {l:,}   [x{l/r:.1f}]")
print()
print("-> 게이트 모델은 RNN 대비 3~4배의 파라미터/계산/메모리.")
print("   -> 데이터가 적으면 '소실을 막아주는' 게이트가 아니라")
print("      '과적합을 부르는' 게이트가 된다. (위 기억 과제의 LSTM<GRU)")

기본 RNN (게이트 1): 82,176   [기준 1x]
GRU      (게이트 3): 246,528   [x3.0]
LSTM     (게이트 4): 328,704   [x4.0]

-> 게이트 모델은 RNN 대비 3~4배의 파라미터/계산/메모리.
   -> 데이터가 적으면 '소실을 막아주는' 게이트가 아니라
      '과적합을 부르는' 게이트가 된다. (위 기억 과제의 LSTM<GRU)


## 6. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| \(0.45^{T-1}\) 직접 계산 | T=50 → \(10^{-17}\), T=100 → \(10^{-35}\) | 소실은 **지수** — 형 자체가 문제 |
| 곱셈 vs 스킵 사슬 (20단계) | \(0.9^{20}\approx0.12\) vs \(1.1^{20}\approx6.7\) | 곱=할인, 덧셈=1을 통과 (ResNet 원리) |
| 셀 그래디언트 \(\prod f_t\) | f=0.99, T=100 → 0.37 (RNN \(10^{-35}\) 대비 \(10^{33}\)×) | 게이트가 승수를 1에 *가깝게* (해결 아님) |
| 기억 과제 3모델×3seed | RNN 0.89 / GRU 0.99 / LSTM 0.86 | 게이트가 기억을 살린다 + GRU>LSTM 반전 |
| 파라미터 수 | GRU≈3×, LSTM≈4× | 게이트 비용 = 편향-분산 트레이드오프 (Ch.6) |

**다음 12장**: 게이트로 소실을 *느리게* 만든 RNN의 근본적 한계(순차성)를
없애는 것이 Attention/Transformer — 순차 압축이 아니라 필요할 때마다 과거
전체를 직접 들여다보는 접근이다.